# TransLLM + Google Colab + Ollama + TranslateGemma

This notebook accompanies my LinkedIn article on running TranslateGemma on a Google Colab GPU and connecting it to Trados Studio through TransLLM.

**Author:** Mohamed Zeid
**LinkedIn:** https://www.linkedin.com/in/arabic/  
**Article:** [LinkedIn article URL]

## Useful links

- TransLLM: https://www.translationapps.com
- Google Colab: https://colab.research.google.com
- Ollama: https://ollama.com
- TranslateGemma on Ollama: https://ollama.com/library/translategemma
- ngrok: https://ngrok.com

## Important

This notebook is intended as a practical example and learning resource.

Do not publish or share your:
- ngrok authentication token
- generated API key
- client text or confidential translation content

The ngrok public URL is generated for the current session and may change when the Colab runtime is restarted.

In [ ]:
!apt-get update -qq
!apt-get install -y zstd -qq

# Correct Ollama installation URL
!curl -fsSL https://ollama.com/install.sh | sh

!pip install -q pyngrok fastapi uvicorn requests

In [ ]:
# ============================================================
# TRANSLATEGEMMA + OLLAMA + SECURE NGROK FOR TRANSLM
# ============================================================

!apt-get update -qq
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh

!pip install -q pyngrok fastapi uvicorn httpx requests


import os
import time
import socket
import secrets
import subprocess
import threading
import requests
import httpx

from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import Response, StreamingResponse
from pyngrok import ngrok
import uvicorn


# ============================================================
# CONFIGURATION
# ============================================================

NGROK_TOKEN = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx" # add your own token

MODEL = "translategemma:12b"

OLLAMA_HOST = "127.0.0.1"
OLLAMA_PORT = 11434
OLLAMA_URL = f"http://{OLLAMA_HOST}:{OLLAMA_PORT}"

# You can replace this with your own key if you prefer
API_KEY = secrets.token_urlsafe(32)


# ============================================================
# START OLLAMA
# ============================================================

print("Starting Ollama...")

env = os.environ.copy()

# IMPORTANT:
# Do NOT expose Ollama itself.
# Only our authenticated proxy will be public.
env["OLLAMA_HOST"] = f"{OLLAMA_HOST}:{OLLAMA_PORT}"

log = open("/tmp/ollama.log", "w")

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=log,
    stderr=subprocess.STDOUT
)


# ============================================================
# WAIT FOR OLLAMA
# ============================================================

print("Waiting for Ollama...")

for i in range(60):

    try:

        r = requests.get(
            f"{OLLAMA_URL}/api/tags",
            timeout=2
        )

        if r.status_code == 200:
            print("Ollama is ready.")
            break

    except requests.RequestException:
        pass

    time.sleep(1)

else:

    raise RuntimeError(
        "Ollama did not start. "
        "Check /tmp/ollama.log"
    )


# ============================================================
# DOWNLOAD TRANSLATEGEMMA
# ============================================================

print(f"\nPulling {MODEL}...")

subprocess.run(
    ["ollama", "pull", MODEL],
    check=True
)

print(f"\n{MODEL} installed.")


# ============================================================
# LOCAL TEST
# ============================================================

print("\nTesting TranslateGemma locally...")

test_payload = {

    "model": MODEL,

    "messages": [
        {
            "role": "user",
            "content":
            """You are a professional English (en) to Arabic (ar) translator.
Translate the following English text into Arabic.
Produce only the Arabic translation with no explanations.


The machine must be switched off before maintenance begins."""
        }
    ],

    "stream": False,

    "options": {
        "temperature": 0.1
    }
}


r = requests.post(
    f"{OLLAMA_URL}/api/chat",
    json=test_payload,
    timeout=600
)


print("Status:", r.status_code)

if r.status_code != 200:

    print(r.text)

    raise RuntimeError(
        "Local Ollama test failed."
    )


result = r.json()

print("\nTranslation:")
print(result["message"]["content"])


# ============================================================
# CREATE OLLAMA-COMPATIBLE SECURE PROXY
# ============================================================

app = FastAPI()


# ------------------------------------------------------------
# AUTHENTICATION
# ------------------------------------------------------------

def verify_key(request: Request):

    authorization = request.headers.get(
        "authorization",
        ""
    )

    x_api_key = request.headers.get(
        "x-api-key",
        ""
    )

    api_key_header = request.headers.get(
        "api-key",
        ""
    )

    valid = False

    # Standard Bearer token
    if authorization == f"Bearer {API_KEY}":
        valid = True

    # Also accept raw token in Authorization header
    if authorization == API_KEY:
        valid = True

    # Alternate API-key headers
    if x_api_key == API_KEY:
        valid = True

    if api_key_header == API_KEY:
        valid = True

    if not valid:

        raise HTTPException(
            status_code=401,
            detail="Invalid API key"
        )


# ============================================================
# ROOT ENDPOINT
# ============================================================

@app.get("/")
async def root():

    return {
        "status": "ok",
        "service": "Ollama TranslateGemma Proxy",
        "model": MODEL
    }


# ============================================================
# OLLAMA NATIVE API PROXY
#
# Supports:
#
# /api/chat
# /api/generate
# /api/tags
# /api/show
# /api/version
# etc.
# ============================================================

@app.api_route(
    "/api/{path:path}",
    methods=[
        "GET",
        "POST",
        "PUT",
        "DELETE"
    ]
)
async def ollama_proxy(
    path: str,
    request: Request
):

    verify_key(request)

    upstream_url = (
        f"{OLLAMA_URL}/api/{path}"
    )

    if request.url.query:

        upstream_url += (
            "?" + request.url.query
        )


    body = await request.body()


    # Copy useful headers but remove host/auth
    headers = {}

    content_type = request.headers.get(
        "content-type"
    )

    if content_type:

        headers["Content-Type"] = content_type


    # --------------------------------------------------------
    # Detect whether request wants streaming
    # --------------------------------------------------------

    wants_stream = False

    try:

        if body:

            data = await request.json()

            wants_stream = bool(
                data.get("stream", False)
            )

    except Exception:

        pass


    # --------------------------------------------------------
    # STREAMING RESPONSE
    # --------------------------------------------------------

    if wants_stream:

        async def stream_generator():

            async with httpx.AsyncClient(
                timeout=None
            ) as client:

                async with client.stream(
                    request.method,
                    upstream_url,
                    headers=headers,
                    content=body
                ) as upstream:

                    async for chunk in (
                        upstream.aiter_bytes()
                    ):

                        yield chunk


        return StreamingResponse(
            stream_generator(),
            media_type="application/x-ndjson"
        )


    # --------------------------------------------------------
    # NORMAL RESPONSE
    # --------------------------------------------------------

    async with httpx.AsyncClient(
        timeout=600
    ) as client:

        upstream = await client.request(
            request.method,
            upstream_url,
            headers=headers,
            content=body
        )


    return Response(
        content=upstream.content,
        status_code=upstream.status_code,
        media_type=upstream.headers.get(
            "content-type",
            "application/json"
        )
    )


# ============================================================
# OPTIONAL OPENAI-COMPATIBLE API
#
# This means the same server can ALSO be used by apps that
# expect:
#
# /v1/models
# /v1/chat/completions
# ============================================================

@app.api_route(
    "/v1/{path:path}",
    methods=[
        "GET",
        "POST"
    ]
)
async def openai_proxy(
    path: str,
    request: Request
):

    verify_key(request)

    upstream_url = (
        f"{OLLAMA_URL}/v1/{path}"
    )

    if request.url.query:

        upstream_url += (
            "?" + request.url.query
        )


    body = await request.body()


    async with httpx.AsyncClient(
        timeout=600
    ) as client:

        upstream = await client.request(
            request.method,
            upstream_url,
            headers={
                "Authorization":
                    "Bearer ollama",

                "Content-Type":
                    request.headers.get(
                        "content-type",
                        "application/json"
                    )
            },
            content=body
        )


    return Response(
        content=upstream.content,
        status_code=upstream.status_code,
        media_type=upstream.headers.get(
            "content-type",
            "application/json"
        )
    )


# ============================================================
# START PROXY
# ============================================================

sock = socket.socket()

sock.bind(
    ("127.0.0.1", 0)
)

PROXY_PORT = (
    sock.getsockname()[1]
)

sock.close()


print(
    f"\nStarting secure proxy "
    f"on port {PROXY_PORT}..."
)


def run_server():

    uvicorn.run(
        app,
        host="127.0.0.1",
        port=PROXY_PORT,
        log_level="warning"
    )


threading.Thread(
    target=run_server,
    daemon=True
).start()


# ============================================================
# WAIT FOR PROXY
# ============================================================

for i in range(30):

    try:

        r = requests.get(
            f"http://127.0.0.1:"
            f"{PROXY_PORT}/",
            timeout=2
        )

        if r.status_code == 200:
            break

    except Exception:
        pass

    time.sleep(1)

else:

    raise RuntimeError(
        "Proxy did not start."
    )


print("Proxy running.")


# ============================================================
# START NGROK
# ============================================================

print("\nStarting ngrok...")

ngrok.kill()

ngrok.set_auth_token(
    NGROK_TOKEN
)

tunnel = ngrok.connect(
    addr=str(PROXY_PORT),
    proto="http"
)

PUBLIC_URL = tunnel.public_url


if PUBLIC_URL.startswith(
    "http://"
):

    PUBLIC_URL = PUBLIC_URL.replace(
        "http://",
        "https://",
        1
    )


# ============================================================
# TEST /api/tags THROUGH NGROK
# ============================================================

print(
    "\nTesting remote Ollama API..."
)


headers = {
    "Authorization":
        f"Bearer {API_KEY}"
}


r = requests.get(
    f"{PUBLIC_URL}/api/tags",
    headers=headers,
    timeout=30
)


print(
    "/api/tags status:",
    r.status_code
)


if r.status_code != 200:

    print(r.text)

    raise RuntimeError(
        "Remote Ollama API test failed."
    )


print("\nModels detected:")

models = r.json().get(
    "models",
    []
)

for model in models:

    print(
        " -",
        model.get("name")
    )


# ============================================================
# REMOTE CHAT TEST
# ============================================================

print(
    "\nTesting /api/chat through ngrok..."
)


r = requests.post(
    f"{PUBLIC_URL}/api/chat",
    headers=headers,
    json=test_payload,
    timeout=600
)


print(
    "/api/chat status:",
    r.status_code
)


if r.status_code != 200:

    print(r.text)

    raise RuntimeError(
        "Remote chat test failed."
    )


print("\nRemote translation:")

print(
    r.json()["message"]["content"]
)


# ============================================================
# PRINT TRANSLM SETTINGS
# ============================================================

print("\n")
print("=" * 70)

print(
    "TRANSLM OLLAMA SETTINGS"
)

print("=" * 70)

print("\nBackend:")
print("Ollama (Cloud)")

print("\nConnection:")
print("Remote server")

print("\nServer URL:")
print(PUBLIC_URL)

print("\nRequires API key:")
print("YES")

print("\nAPI key:")
print(API_KEY)

print("\nModel Name:")
print(MODEL)

print("\n")
print("=" * 70)

print(
    "IMPORTANT: DO NOT ADD /v1 "
    "OR /api/chat TO SERVER URL"
)

print("=" * 70)